In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import data

In [7]:
root = "../../dataset/CIFAKE/"
model_names = ['ResNet', 'MobileNet', 'ViT16']

label_to_class = data.get_id_to_class()
target_names = list(label_to_class.values())

# Fake - Real Analysis

In [8]:
train_setting = 'fake'
test_setting = 'real'
embeds = ['CLIP', 'DINOv2', 'DINOv3']
reducers = ['UMAP', 'TSNE']

In [9]:
def save_distance_dataframe(root, df, model_name, embed, reducer):

    clip_filepath = f'{root}distance/dis_{embed}_{reducer}_train.csv'
    df_clip = pd.read_csv(clip_filepath)
    df = pd.merge(df, df_clip, on=['label'], how='left', suffixes=('', '_red'))
    
    embed_filepath = f'{root}distance/embed_{embed}_{reducer}_train.csv'
    df_embed = pd.read_csv(embed_filepath)
    df = pd.merge(df, df_embed, on='filepath', how='left', suffixes=('', '_embed'))
    
    df["distance from real"] = ((df["embeddings x"] - df["r centroid x"])**2 + 
                                    (df["embeddings y"] - df["r centroid y"])**2) ** 0.5
    df["inside circle"] = df["distance from real"] <= df["r radius"]
    
    save_filepath = f'{root}results/{model_name}_{embed}_{reducer}_train.csv'
    df.to_csv(save_filepath, index=False)

In [11]:
for model_name in model_names:
    filepath = f'{root}train.csv'
    df_train = pd.read_csv(filepath)

    for embed in embeds:
        for reducer in reducers:
            save_distance_dataframe(root, df_train, model_name, embed, reducer)